# 🎙️ أكاديمية إتقان — صوتك لكل المحتوى (مجانًا على Google Colab)

الدفتر ده بيعمل نسخة من صوتك من المقاطع اللي سجّلتها، ويقرا بيها كل جمل المنهج (غير القرآن)، ويرفعها على الموقع أوتوماتيك.
الصوت المستنسخ بيتشغّل بدل الصوت الآلي، والتسجيل البشري بصوتك الحقيقي بيفضل ليه الأولوية.

**⚠️ القرآن عمره ما بيتولّد بالذكاء الاصطناعي.** الدفتر ده للمنهج والشرح والمحادثات بس.

### الخطوات (مرة واحدة):
1. من فوق: **Runtime ← Change runtime type ← T4 GPU** (مجاني).
2. شغّل الخلايا بالترتيب (▶ على كل خلية).
3. في خلية ٢ ارفع: مقطع أو اتنين بصوتك (١٠–٣٠ ثانية، مكان هادي) + ملف `itqan-voice-lines.csv` (من الموقع: لوحة التحكم ← الإعدادات ← محاكاة الأكاديمية ← «صدّر جمل المنهج»).
4. في الآخر هيتنزل ملف `itqan-voices.zip` — ارفعه من الموقع: محاكاة الأكاديمية ← «صوتك في المحتوى» ← «ارفع ملف الأصوات». **مفيش أي باسورد.**

### الأداة المستخدمة
**Chatterbox Multilingual** من Resemble AI (ترخيص MIT — مسموح تجاريًا، وبيدعم العربي).
لو حصل خطأ في التثبيت أو اسم الدالة اتغيّر، راجع صفحة المكتبة: https://github.com/resemble-ai/chatterbox


In [ ]:
# 1) التثبيت (3–5 دقايق)
!pip -q install chatterbox-tts supabase pydub librosa soundfile
!nvidia-smi -L

In [ ]:
# 2) ارفع مقاطع صوتك + ملف الجمل
from google.colab import files
up = files.upload()   # اختار ملفات صوتك (mp3/wav/m4a) + itqan-voice-lines.csv
refs = [n for n in up if not n.endswith('.csv')]
csv_name = [n for n in up if n.endswith('.csv')][0]
print('voice refs:', refs, '| lines file:', csv_name)

In [ ]:
# 3) جهّز مرجع صوتك (أول 25 ثانية نضيفة) وحمّل الموديل
import torch, torchaudio, librosa, soundfile as sf, numpy as np
from pydub import AudioSegment
seg = AudioSegment.empty()
for r in refs: seg += AudioSegment.from_file(r).set_channels(1).set_frame_rate(24000)
seg = seg[:25000]; seg.export('ref.wav', format='wav')
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
model = ChatterboxMultilingualTTS.from_pretrained(device='cuda' if torch.cuda.is_available() else 'cpu')
wav = model.generate('أهلًا بيك في أكاديمية إتقان', language_id='ar', audio_prompt_path='ref.wav')
torchaudio.save('test.wav', wav, model.sr)
from IPython.display import Audio; Audio('test.wav')  # اسمع: لو مش شبهك، جرّب مقطع أنضف

### 4) مفيش تسجيل دخول — الأصوات هتتجمّع في ملف واحد ترفعه من الموقع

In [ ]:
# 5) ولّد كل الجمل في ملف مضغوط (يكمل من مكان ما وقف لو قطع)
import csv, hashlib, json, os, zipfile, time
from pydub import AudioSegment
import soundfile as sf, librosa
PITCH = {'m': 0, 'f': 4, 'k': 7}   # الست والطفل = صوتك بطبقة أعلى
LIMIT = 400                        # عدد الجمل في المرة
def h(t): return hashlib.sha256(t.strip().encode()).hexdigest()[:40]
rows = list(csv.DictReader(open(csv_name, encoding='utf-8-sig')))
os.makedirs('out', exist_ok=True)
man = json.load(open('out/manifest.json')) if os.path.exists('out/manifest.json') else []
done = {(x['h'], x['kind']) for x in man}
todo = [x for x in rows if (h(x['text']), x['kind']) not in done][:LIMIT]
print(len(rows), 'lines total |', len(done), 'done before |', len(todo), 'now')
for i, x in enumerate(todo):
    t, kind = x['text'].strip(), x['kind']
    try:
        w = model.generate(t, language_id='ar', audio_prompt_path='ref.wav').squeeze().cpu().numpy()
        if PITCH.get(kind): w = librosa.effects.pitch_shift(w, sr=model.sr, n_steps=PITCH[kind])
        sf.write('o.wav', w, model.sr); f = f"{h(t)}-{kind}.mp3"
        AudioSegment.from_wav('o.wav').export('out/' + f, format='mp3', bitrate='64k')
        man.append({'h': h(t), 'text': t, 'kind': kind, 'file': f})
        if i % 20 == 0:
            print(i, t[:40]); json.dump(man, open('out/manifest.json', 'w'), ensure_ascii=False)
    except Exception as e:
        print('skip', t[:30], e); time.sleep(1)
json.dump(man, open('out/manifest.json', 'w'), ensure_ascii=False)
with zipfile.ZipFile('itqan-voices.zip', 'w') as z:
    for x in man: z.write('out/' + x['file'], x['file'])
    z.write('out/manifest.json', 'manifest.json')
from google.colab import files; files.download('itqan-voices.zip')
print('✓ اتنزّل itqan-voices.zip — ارفعه من الموقع')

---
# 🖼️ (اختياري) صور بالذكاء الاصطناعي مجانًا

بيولّد صورة لأي مفتاح من صور الموقع (زي `surah:1:0`) بموديل **FLUX.1-schnell** (ترخيص Apache-2.0 مسموح تجاريًا)، ويرفعها مكان القديمة.
⚠️ من غير بشر ولا وجوه — طبيعة ومناظر بس.

In [ ]:
!pip -q install diffusers transformers accelerate sentencepiece
import torch, os, zipfile, json
from diffusers import FluxPipeline
pipe = FluxPipeline.from_pretrained('black-forest-labs/FLUX.1-schnell', torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
os.makedirs('art', exist_ok=True); art = []
def make(key, prompt):
    img = pipe(prompt + ', no people, no faces, no text, cinematic light, ultra detailed', num_inference_steps=4, guidance_scale=0.0, height=768, width=1344).images[0]
    f = key.replace(':', '_') + '.jpg'; img.save('art/' + f, quality=88); art.append({'key': key, 'file': f, 'prompt': prompt}); return img
make('surah:1:0', 'golden sunrise over calm desert dunes, islamic geometric arches in the distance')
with zipfile.ZipFile('itqan-art.zip', 'w') as z:
    for a in art: z.write('art/' + a['file'], a['file'])
    z.writestr('manifest.json', json.dumps(art, ensure_ascii=False))
from google.colab import files; files.download('itqan-art.zip')